# 03 - Baselines

Establish performance baselines:
1. **BLAST** — the standard in marine biology
2. **k-NN on k-mer frequencies** — simple ML baseline
3. **Random Forest on k-mers** — stronger ML baseline

These are the numbers MarineMamba needs to beat.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kluless13/marinemamba/blob/main/notebooks/03_baselines.ipynb)

In [ ]:
# === COLAB SETUP ===
!pip install pandas scikit-learn biopython tqdm
!apt-get install -y ncbi-blast+
!git clone https://github.com/kluless13/marinemamba.git
%cd marinemamba

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import json
import subprocess
import tempfile
import time
from pathlib import Path
from collections import Counter
from itertools import product
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from tqdm import tqdm

# Google Drive paths
PROC_DIR = Path("/content/drive/MyDrive/marinemamba/data/processed")
RESULTS_DIR = Path("/content/drive/MyDrive/marinemamba/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(PROC_DIR / "supervised_train.csv")
test = pd.read_csv(PROC_DIR / "supervised_test.csv")
unseen = pd.read_csv(PROC_DIR / "unseen.csv")

print(f"Train: {len(train)} | Test: {len(test)} | Unseen: {len(unseen)}")

## 1. BLAST Baseline

In [ ]:
def run_blast_baseline(train_df, test_df, label_col="species_name"):
    """Run BLASTn: build DB from train, query with test, measure accuracy."""
    with tempfile.TemporaryDirectory() as tmpdir:
        # Write train as FASTA (reference DB)
        db_fasta = f"{tmpdir}/db.fasta"
        with open(db_fasta, "w") as f:
            for idx, row in train_df.iterrows():
                f.write(f">{idx}|{row[label_col]}\n{row['nucleotides']}\n")
        
        # Write test as FASTA (queries)
        query_fasta = f"{tmpdir}/query.fasta"
        with open(query_fasta, "w") as f:
            for idx, row in test_df.iterrows():
                f.write(f">{idx}\n{row['nucleotides']}\n")
        
        # Build BLAST DB
        subprocess.run(
            ["makeblastdb", "-in", db_fasta, "-dbtype", "nucl", "-out", f"{tmpdir}/blastdb"],
            capture_output=True, check=True
        )
        
        # Run BLASTn
        start_time = time.time()
        result = subprocess.run(
            ["blastn", "-query", query_fasta, "-db", f"{tmpdir}/blastdb",
             "-outfmt", "6 qseqid sseqid pident evalue", "-max_target_seqs", "1",
             "-evalue", "1e-5", "-num_threads", "4"],
            capture_output=True, text=True, check=True
        )
        elapsed = time.time() - start_time
        
        # Parse results
        predictions = {}
        for line in result.stdout.strip().split("\n"):
            if not line:
                continue
            parts = line.split("\t")
            query_id = int(parts[0])
            subject_label = parts[1].split("|", 1)[1] if "|" in parts[1] else "Unknown"
            if query_id not in predictions:  # Keep top hit only
                predictions[query_id] = subject_label
        
        # Calculate accuracy
        y_true = []
        y_pred = []
        no_hit = 0
        for idx, row in test_df.iterrows():
            y_true.append(row[label_col])
            if idx in predictions:
                y_pred.append(predictions[idx])
            else:
                y_pred.append("NO_HIT")
                no_hit += 1
        
        acc = accuracy_score(y_true, y_pred)
        seqs_per_sec = len(test_df) / elapsed
        
        return {
            "accuracy": acc,
            "no_hit_rate": no_hit / len(test_df),
            "seqs_per_sec": seqs_per_sec,
            "elapsed_sec": elapsed
        }

print("Running BLAST baseline on test set...")
blast_results = run_blast_baseline(train, test, label_col="species_name")
print(f"BLAST species accuracy: {blast_results['accuracy']:.4f}")
print(f"BLAST no-hit rate: {blast_results['no_hit_rate']:.4f}")
print(f"BLAST speed: {blast_results['seqs_per_sec']:.1f} seqs/sec")

## 2. k-mer Feature Extraction

In [ ]:
def extract_kmer_features(sequences, k=6):
    """Extract k-mer frequency vectors from DNA sequences."""
    # All possible k-mers
    all_kmers = [''.join(p) for p in product('ACGT', repeat=k)]
    kmer_to_idx = {km: i for i, km in enumerate(all_kmers)}
    
    features = np.zeros((len(sequences), len(all_kmers)), dtype=np.float32)
    
    for i, seq in enumerate(tqdm(sequences, desc=f"Extracting {k}-mers")):
        counts = Counter()
        for j in range(len(seq) - k + 1):
            kmer = seq[j:j+k]
            if kmer in kmer_to_idx:  # Skip k-mers with N
                counts[kmer] += 1
        total = sum(counts.values())
        if total > 0:
            for kmer, count in counts.items():
                features[i, kmer_to_idx[kmer]] = count / total
    
    return features

print("Extracting 6-mer features...")
X_train = extract_kmer_features(train["nucleotides"].tolist(), k=6)
X_test = extract_kmer_features(test["nucleotides"].tolist(), k=6)
X_unseen = extract_kmer_features(unseen["nucleotides"].tolist(), k=6)

print(f"Feature shape: {X_train.shape}")

## 3. k-NN Baseline

In [ ]:
# Species-level k-NN on test set
knn = KNeighborsClassifier(n_neighbors=1, metric="cosine", n_jobs=-1)
knn.fit(X_train, train["species_name"])

start = time.time()
knn_species_pred = knn.predict(X_test)
knn_elapsed = time.time() - start

knn_species_acc = accuracy_score(test["species_name"], knn_species_pred)
print(f"k-NN species accuracy (test): {knn_species_acc:.4f}")
print(f"k-NN speed: {len(X_test)/knn_elapsed:.1f} seqs/sec")

# Genus-level k-NN on unseen genera
knn_genus = KNeighborsClassifier(n_neighbors=1, metric="cosine", n_jobs=-1)
knn_genus.fit(X_train, train["genus_name"])
knn_genus_pred = knn_genus.predict(X_unseen)
knn_genus_acc = accuracy_score(unseen["genus_name"], knn_genus_pred)
knn_genus_bacc = balanced_accuracy_score(unseen["genus_name"], knn_genus_pred)
print(f"k-NN genus accuracy (unseen): {knn_genus_acc:.4f} (balanced: {knn_genus_bacc:.4f})")

## 4. Random Forest Baseline

In [ ]:
rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
rf.fit(X_train, train["species_name"])

rf_species_pred = rf.predict(X_test)
rf_species_acc = accuracy_score(test["species_name"], rf_species_pred)
print(f"Random Forest species accuracy (test): {rf_species_acc:.4f}")

## 5. Save Results

In [ ]:
baselines = {
    "blast": {
        "species_accuracy": blast_results["accuracy"],
        "no_hit_rate": blast_results["no_hit_rate"],
        "seqs_per_sec": blast_results["seqs_per_sec"]
    },
    "knn_1nn_cosine": {
        "species_accuracy_test": float(knn_species_acc),
        "genus_accuracy_unseen": float(knn_genus_acc),
        "genus_balanced_accuracy_unseen": float(knn_genus_bacc)
    },
    "random_forest": {
        "species_accuracy_test": float(rf_species_acc)
    }
}

with open(RESULTS_DIR / "baselines.json", "w") as f:
    json.dump(baselines, f, indent=2)

print("\n" + "=" * 60)
print("BASELINE SUMMARY")
print("=" * 60)
print(json.dumps(baselines, indent=2))
print("=" * 60)

## Next

Proceed to `04_pretrain.ipynb` for domain-adaptive pretraining (requires GPU).